# 📊 CEM4644 · MP6 — Tables and time series
## Workshop (in class): *Concrete mixes, and four buildings' electricity*

**No coding needed.** Each grey box is one step: click ▶, wait, read the result, answer the report question. Run from top to bottom.

Photos and drawings were the last four labs. Most construction data is neither: it is a **table** (one row per mix, per building, per bid) or a **time series** (one value per hour, per day). This lab does the same things with them: predict a number, predict a class, forecast what comes next, and check every answer against what really happened. About 90 minutes. No GPU needed.

In [ ]:
#@title ▶ Step 0 · Run me first (1–2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Untick *load_forecaster* to skip the pretrained forecasting model (Step 5 then has two methods instead of three).
load_forecaster = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp6_tabular_timeseries", "aec_tab"
FOLDERS = ["mp6_tabular_timeseries"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_tab import lab
lab.setup(dataset="workshop", load_forecaster=load_forecaster)


## Part 1 · A table

1,030 concrete mixes tested in a laboratory: what went into each cubic metre, how old the sample was, and the strength it reached. Every row is one mix; the last column, **compressive strength (MPa)**, is the answer the model will learn to predict from the others.

In [ ]:
#@title ▶ Step 1a · Look at the table { display-mode: "form" }
rows = 10 #@param [5, 10, 20] {type:"raw"}
lab.show_table(rows)


In [ ]:
#@title ▶ Step 1b · Guess it yourself { display-mode: "form" }
#@markdown Five mixes without their answer. Type your guess for each and click the button; Step 2a shows what the model makes of the same mixes.
lab.guess()


## Part 2 · Two questions, one table

The same table can answer **how much?** (a number: *regression*) or **which class?** (a category: *classification*). Both models train on 80 % of the rows and are scored on the 20 % they never saw.

In [ ]:
#@title ▶ Step 2a · Regression: predict the number { display-mode: "form" }
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
lab.regression(model)


In [ ]:
#@title ▶ Step 2b · Classification: predict the class { display-mode: "form" }
#@markdown *grades* puts each mix in one of 3 bands of compressive strength; *pass / fail* asks whether it reaches the *threshold* you set.
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
task = "grades" #@param ["grades", "pass / fail against a specification"]
threshold = 30 #@param {type:"slider", min:15, max:60, step:5}
lab.classification(model, task, threshold)


> ### 📝 Report question 1
> From Step 2a: the average miss of the straight line and of the trees, in MPa, and what the worst misses have in common. From Step 2b: how many mixes land in the right grade, and at your pass / fail threshold how many false passes and false fails there are. What is the difference between predicting 33 MPa and predicting 'pass', and which of the two mistakes costs more on a real project?

## Part 3 · What the model learned

A model that scores well may still have learned the wrong thing. Two checks: which columns it leans on, and how its prediction moves when you change one input at a time.

In [ ]:
#@title ▶ Step 3a · Which columns matter { display-mode: "form" }
lab.importance()


In [ ]:
#@title ▶ Step 3b · What if… { display-mode: "form" }
#@markdown Move a slider; the prediction updates. Everything not on a slider stays as it is in the chosen row.
start_from = "a typical row" #@param ["a typical row", "row 12", "row 100", "row 500"]
lab.whatif(start_from)


> ### 📝 Report question 2
> From Step 3: the three columns that matter most. Does the model agree with what you know about concrete (more water, longer curing, more cement)? Push one slider to the edge of its range: where does the prediction stop making sense, and why can a model not know that?

## Part 4 · A time series

Four buildings, hourly electricity in 2017: electricity, hour by hour, from 4 real buildings on North American campuses, with the site's air temperature. A time series has a **rhythm** (days, weeks, seasons) that a table does not, and a model that knows the rhythm can say what comes next.

In [ ]:
#@title ▶ Step 4a · Which building is which? { display-mode: "form" }
#@markdown Four buildings, no names: a week and a year each. They are, in some order, assembly hall, office, residence hall, school.
lab.buildings()


In [ ]:
#@title ▶ Step 4b · Your answer { display-mode: "form" }
a = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
b = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
c = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
d = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
lab.buildings_answer(a, b, c, d)


In [ ]:
#@title ▶ Step 4c · The anatomy of one building's year { display-mode: "form" }
building = "Hog_office_Marlena: office, 12,240 m²" #@param ["Hog_office_Marlena: office, 12,240 m²", "Bear_education_Lila: school, 11,484 m²", "Bear_lodging_Evan: residence hall, 17,206 m²", "Bear_assembly_Jose: assembly hall, 3,361 m²"]
lab.anatomy(building)


> ### 📝 Report question 3
> From Step 4: which buildings did you get right, and from what (the shape of the day, the weekend, the summer)? Pick one building in Step 4c and describe its week in three sentences a facilities manager would recognise.

## Part 5 · Next week

Three ways to forecast a week: copy last week; decision trees that learned from the past weeks, the calendar and the temperature; and a **pretrained forecasting model** that has seen millions of other time series and none of ours. Each is scored against what really happened.

In [ ]:
#@title ▶ Step 5a · Forecast one week { display-mode: "form" }
building = "Hog_office_Marlena: office, 12,240 m²" #@param ["Hog_office_Marlena: office, 12,240 m²", "Bear_education_Lila: school, 11,484 m²", "Bear_lodging_Evan: residence hall, 17,206 m²", "Bear_assembly_Jose: assembly hall, 3,361 m²"]
method = "all three" #@param ["all three", "same hour last week", "decision trees (last weeks + calendar + temperature)", "Chronos-Bolt (a pretrained forecasting model, zero-shot)"]
lab.forecast(building, method)


> ### 📝 Report question 4
> From Step 5a on all four buildings: the average miss of each method (copy the tables). Which method wins where, and is 'same hour last week' ever hard to beat? What does the shaded band of the pretrained model mean, and how would you use it when planning a site's power supply?

## Part 6 · The odd days

Every building has a usual day for each weekday. A day that leaves the pattern is either explained (a holiday, a closure) or worth a phone call (a fault, a meter, something left running).

In [ ]:
#@title ▶ Step 6a · Days that do not fit { display-mode: "form" }
#@markdown Lower the threshold and more days are flagged; raise it and only the strangest remain.
building = "Hog_office_Marlena: office, 12,240 m²" #@param ["Hog_office_Marlena: office, 12,240 m²", "Bear_education_Lila: school, 11,484 m²", "Bear_lodging_Evan: residence hall, 17,206 m²", "Bear_assembly_Jose: assembly hall, 3,361 m²"]
threshold = 3.5 #@param {type:"slider", min:2, max:6, step:0.5}
lab.odd_days(building, threshold)


> ### 📝 Report question 5
> From Step 6a on two buildings: the flagged days at threshold 3.5. Which have an obvious cause (the calendar column), which do not? For one unexplained day, say what you would check first. What threshold would you set for an automatic alert, and why?

## Part 7 · Your own table or time series

A small app, opened from a link: upload any CSV. A table gets decision trees and a score on held-out rows; a time series gets a forecast of its last period.

In [ ]:
#@title ▶ Step 7 · Your own data { display-mode: "form" }
#@markdown Open the printed link in a new tab.
lab.upload_app()


> ### 📝 Report question 6
> Upload one table or one time series of your own (a cost table, a utility bill history, anything with numbers) and report what the app found: the score, the columns that mattered or the forecast, and whether you believe it.

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Table: Concrete Compressive Strength, I-Cheng Yeh (1998), UCI Machine Learning Repository, CC BY 4.0, https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength.
- Meters: Building Data Genome Project 2 (hourly electricity meters and site weather, 2017), Miller et al. (2020), Scientific Data 7:368, MIT, https://github.com/buds-lab/building-data-genome-project-2.
- Pretrained forecaster: Chronos-Bolt (small) (Amazon Science, Apache-2.0).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp6_tabular_timeseries`).